<a href="https://colab.research.google.com/github/aims-ai-research-foundations/pilot-workshop/blob/main/assignments/day5/day5-course7-student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Course 07: Accelerate Your Model Lab (Student Practical Notebook)

**💡 Can you get this model running on a single GPU?**

In this notebook, you will start by exploring what happens when you scale up a language model. You will see that bigger models produce better output, but they cost more to train and take more memory to run. You will then try to load a model that does not fit on your GPU, figure out exactly why it failed, and learn two techniques for working within memory constraints.

**⏱️ Estimated time:** 60 minutes

**🔄 Protocol: Predict → Run → Interpret**

For every code cell in this notebook, you must:
1. **🔮 Predict**: Write what you think will happen before running the cell.
2. **▶️ Run**: Execute the cell and observe the actual output.
3. **🔍 Interpret**: Explain why the output matches or differs from your prediction. If your prediction was wrong, that is where the learning happens.

---

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. Compare the output quality of transformer models of different sizes and observe the relationship between scale and performance.
2. Estimate the training FLOPs required for a transformer model using the $6 \times N \times D$ approximation (forward: $2 \times N \times D$, backward: $4 \times N \times D$).
3. Identify what happens when a model exceeds available GPU memory and explain why it fails.
4. Estimate GPU memory usage by calculating the memory required for parameters, optimizer states, gradients, and activations.
5. Explain how quantization (bfloat16, int8) reduces memory usage and describe the trade-offs against float32.
6. Explain how gradient accumulation achieves a larger effective batch size without increasing memory usage and describe the trade-off between memory and training time.

---

## Notebook Structure

| Section | Title | What You Will Do | Time |
|---|---|---|---|
| 1 | Compare Models of Different Sizes | Generate text from a 270M and 1B model. See that bigger is better. Ask: but at what cost? | 10 min |
| 2 | Estimate Training FLOPs | Implement the 6ND formula. Discover that compute scales linearly with both parameters and tokens. | 10 min |
| 3 | Hitting a Wall | Attempt to load a 4B model in float32. It crashes. This failure motivates everything that follows. | 5 min |
| 4 | Estimate GPU Memory | Calculate the memory breakdown (parameters + optimizer + gradients + activations). Understand why it crashed. | 15 min |
| 5 | Precision Formats and Quantization | Learn how bfloat16 and int8 reduce memory. Calculate which models fit at each precision. | 10 min |
| 6 | Gradient Accumulation | Learn how to simulate large batch sizes without the memory cost. Calculate the trade-off between memory and time. | 10 min |

---

**📚 Reference:** [github.com/google-deepmind/ai-foundations/tree/main/course_7](https://github.com/google-deepmind/ai-foundations/tree/main/course_7)

---

## Install Dependencies

Run the cell below to install and import all packages used in this notebook.

- **keras**: High-level deep learning framework. We use the JAX backend for performance.
- **keras-hub**: Load pretrained Gemma models from Kaggle using `from_preset()`.
- **plotly**: Create interactive charts and visualisations.
- **pandas**: Structure and display tabular data.
- **numpy**: Numerical operations and array manipulation.

In [1]:
# 📦 Install and import all dependencies
# Just run this cell. Do not modify it.

!pip install -q -U keras-hub keras

import os

# Set backend BEFORE importing keras
os.environ["KERAS_BACKEND"] = "jax"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.00"

import keras
import keras_hub
import numpy as np
import pandas as pd
import time
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print(f"✅ Keras version: {keras.__version__}")
print(f"✅ Backend: {keras.backend.backend()}")
print("✅ All dependencies installed and imported.")

✅ Keras version: 3.14.1
✅ Backend: jax
✅ All dependencies installed and imported.


---

## Kaggle Setup

To download Gemma models, you need a Kaggle account and API key.

**If you have not done this before:**
1. Go to [kaggle.com/settings](https://www.kaggle.com/settings) and click **Create New Token** under the API section. This downloads a `kaggle.json` file.
2. In Colab, click the **🔑 Secrets** icon in the left sidebar.
3. Add two secrets:
   - `KAGGLE_USERNAME`: your Kaggle username
   - `KAGGLE_KEY`: the API key from the downloaded file
4. Toggle the switch to make both secrets visible to this notebook.

> ⚠️ You must also accept the Gemma model license on Kaggle before downloading. Visit the [Gemma model page on Kaggle](https://www.kaggle.com/models/google/gemma-3) and click **Request Access** if you have not already.

In [2]:
# 🔑 Set Kaggle credentials
# Just run this cell. Do not modify it.

from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

print(f"✅ Kaggle authenticated as: {os.environ['KAGGLE_USERNAME']}")

✅ Kaggle authenticated as: simiokunowo


---

## ⚙️ GPU Check

This notebook requires a GPU runtime. If you have not enabled one yet:
1. Go to **Runtime** in the top menu bar.
2. Click **Change runtime type**.
3. Under **Hardware accelerator**, select **T4 GPU**.
4. Click **Save**.

> ⚠️ Do not proceed until you have a GPU runtime enabled. Run the cell below to confirm.

In [3]:
# ✅ Check GPU availability
# Just run this cell. Do not modify it.

import jax

devices = jax.devices()
print(f"✅ JAX devices: {devices}")

# Updated check to include 'cuda' which is how JAX labels GPUs
if any(label in str(d).lower() for d in devices for label in ["gpu", "cuda"]):
    print(f"✅ GPU detected. Ready to proceed.")
else:
    raise RuntimeError(
        "❌ No GPU detected!\n"
        "Go to Runtime → Change runtime type → Select T4 GPU, "
        "then restart and run all cells again."
    )

✅ JAX devices: [CudaDevice(id=0)]
✅ GPU detected. Ready to proceed.


> The T4 GPU has 16 GB of total memory, but roughly 1 GB is reserved in Google Colab, leaving approximately 15 GB usable for your models.

---

## Section 1: Compare Models of Different Sizes

⏱️ **10 minutes** | 🔮 Predict → ▶️ Run → 🔍 Interpret

**The question:** Does a bigger model actually produce better output?
And if so, what does "bigger" cost you?

In this section, you will load two Gemma 3 models (270M and 1B parameters)
and generate text from the same prompt.
All the code is provided. Your job is to observe, not to write code yet.

> **📝 Before you run anything**, write down your predictions:
>
> - If you give a 270M and a 1B parameter model the same prompt,
>   how will the outputs differ? (Length? Coherence? Accuracy?)
> - How much more memory do you think the 1B model will use?
> - Will the 1B model be noticeably slower to generate text?

**🔮 Your prediction:** *(double-click to edit)*

- Output quality:
- Memory difference:
- Speed difference:

In [4]:
# 🎯 Set your prompt
# Try different prompts to see how the models respond.

prompt = "The key challenge in deploying large language models at African universities is"  # @param {type:"string"}
max_tokens = 200  # @param {type:"slider", min:50, max:500, step:50}

print(f"Prompt: {prompt}")
print(f"Max output tokens: {max_tokens}")

Prompt: The key challenge in deploying large language models at African universities is
Max output tokens: 200


### Load the Gemma 3 270M (270 million parameters) model

In [5]:
print("=" * 60 + "\nMODEL: Gemma 3 270M (instruction-tuned)\n" + "=" * 60)

gemma_270m = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_instruct_270m")
gemma_270m.summary()

MODEL: Gemma 3 270M (instruction-tuned)


Preprocessor: "gemma3_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma3_tokenizer (Gemma3Tokenizer)                            │                      Vocab size: 262,144 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma3_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma3_backbone               │ (None, None, 640)         │     268,098,176 │ padding_mask[0][0],        │
│ (Gemma3Backbone)              │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 262144)      │     167,772,160 │ gemma3_backbone[0][0]      │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 268,098,176 (1022.71 MB)

 Trainable params: 268,098,176 (1022.71 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
start_time = time.time()
output_270m = gemma_270m.generate(prompt, max_length=max_tokens)
time_270m = time.time() - start_time

params_270m = gemma_270m.count_params()
memory_270m = params_270m * 4 / 1e9

print("\n" + "=" * 60 + "\nOUTPUT\n" + "=" * 60)
print(output_270m)
print("\n" + "=" * 60 + "\nSTATS\n" + "=" * 60)
print(f"Parameters:                  {params_270m:,}")
print(f"Parameter memory (float32):  {memory_270m:.2f} GB")
print(f"Generation time:             {time_270m:.2f} seconds")


OUTPUT
The key challenge in deploying large language models at African universities is ensuring the quality, reliability, and accessibility of the models. This challenge is exacerbated by the limited resources, infrastructure, and expertise available to African institutions. This paper explores various strategies to address this challenge, focusing on the need for a holistic approach that integrates various technical, pedagogical, and organizational factors. We will examine the existing challenges and propose a framework for developing and deploying large language models in African universities. The paper will also discuss the potential benefits of using large language models in African universities, including improved research capabilities, enhanced student engagement, and the development of new skills in AI. Finally, we will outline the key considerations for implementing large language models in African universities, including data privacy, ethical considerations, and the need for 

### Loading the next model

You just loaded Gemma 3 270M and used it to generate a response.
This model has approximately 270 million parameters,
which at float32 precision takes around 1 GB of GPU memory for the weights.

Now we want to do the same thing with the larger 1B model.

> We need to free the memory occupied by the 270M model first,
because both models cannot fit in GPU memory at the same time.

In [7]:
# Free memory before loading the next model
del gemma_270m
keras.backend.clear_session()
print("✅ Memory cleared. Ready to load the 1B model.")

✅ Memory cleared. Ready to load the 1B model.


In [8]:
# Load Gemma 3 1B model
gemma_1b = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_instruct_1b")
gemma_1b.summary()

Preprocessor: "gemma3_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma3_tokenizer (Gemma3Tokenizer)                            │                      Vocab size: 262,144 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma3_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma3_backbone               │ (None, None, 1152)        │     999,885,952 │ padding_mask[0][0],        │
│ (Gemma3Backbone)              │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 262144)      │     301,989,888 │ gemma3_backbone[0][0]      │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 999,885,952 (3.72 GB)

 Trainable params: 999,885,952 (3.72 GB)

 Non-trainable params: 0 (0.00 B)

In [9]:
start_time = time.time()
output_1b = gemma_1b.generate(prompt, max_length=max_tokens)
time_1b = time.time() - start_time

params_1b = gemma_1b.count_params()
memory_1b = params_1b * 4 / 1e9

print("\n" + "=" * 60 + "\nOUTPUT\n" + "=" * 60)
print(output_1b)
print("\n" + "=" * 60 + "\nSTATS\n" + "=" * 60)
print(f"Parameters:                  {params_1b:,}")
print(f"Parameter memory (float32):  {memory_1b:.2f} GB")
print(f"Generation time:             {time_1b:.2f} seconds")


OUTPUT
The key challenge in deploying large language models at African universities is the lack of adequate infrastructure, including reliable internet connectivity, sufficient computing power, and skilled personnel.

Here's a breakdown of the key challenges and potential solutions:

**1. Infrastructure Challenges:**

* **Limited Internet Connectivity:** Many universities in Africa struggle with unreliable or slow internet speeds, hindering the ability to run large models.
* **Insufficient Computing Power:**  Large language models require significant processing power, which is often unavailable or expensive in many African institutions.
* **Data Storage:**  Large language models require massive datasets for training and inference, which can be difficult to store and manage.

**2. Skills Gap:**

* **Lack of Expertise:** There's a shortage of data scientists, machine learning engineers, and NLP specialists with the necessary expertise to develop, deploy, and maintain these models.
* **L

### 📊 Gemma 3 270M vs 1B

The charts below compare the two models across three dimensions:
parameters, memory (both inference and estimated training), and generation time.
The red dashed line shows the 16 GB limit of a T4 GPU.

In [46]:
#@title Run this cell to generate the comparison charts { display-mode: "form" }

models = ["Gemma 270M", "Gemma 1B"]
param_counts = [params_270m / 1e9, params_1b / 1e9]
memory_values = [memory_270m, memory_1b]
time_values = [time_270m, time_1b]
training_memory = [m * 4 for m in memory_values]
colors = ["#2D1768", "#F7B733"]

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        "Parameters (billions)",
        "Memory: Inference vs Training (GB)",
        "Generation Time (s)"
    ),
    horizontal_spacing=0.13
)

fig.add_trace(go.Bar(
    x=models, y=param_counts, marker_color=colors,
    text=[f"{v:.2f}B" for v in param_counts],
    textposition="outside", showlegend=False, width=0.4
), row=1, col=1)

fig.add_trace(go.Bar(
    name="Inference (params only)",
    x=models, y=memory_values, marker_color=colors,
    text=[f"{v:.1f} GB" for v in memory_values],
    textposition="outside", width=0.3
), row=1, col=2)

fig.add_trace(go.Bar(
    name="Training (~4x params)",
    x=models, y=training_memory,
    marker_color=["#6B4FCF", "#D49A1A"],
    text=[f"{v:.1f} GB" for v in training_memory],
    textposition="outside", width=0.3
), row=1, col=2)

fig.add_hline(
    y=16, line_dash="dash", line_color="red",
    annotation_text="16 GB GPU limit",
    annotation_position="top left", row=1, col=2
)

fig.add_trace(go.Bar(
    x=models, y=time_values, marker_color=colors,
    text=[f"{v:.1f}s" for v in time_values],
    textposition="outside", showlegend=False, width=0.4
), row=1, col=3)

fig.update_yaxes(range=[0, max(param_counts) * 1.25], row=1, col=1)
fig.update_yaxes(range=[0, max(training_memory) * 1.25], row=1, col=2)
fig.update_yaxes(range=[0, max(time_values) * 1.25], row=1, col=3)

fig.update_layout(
    title=dict(
        text="<b>Gemma 270M vs 1B: What Does Scale Cost You?</b>",
        y=0.98, x=0.5, xanchor="center", yanchor="top"
    ),
    height=450, width=1050,
    template="plotly_white",
    barmode="group",
    margin=dict(t=110, b=80, l=50, r=30),
    legend=dict(
        orientation="h", yanchor="top", y=1.3,
        xanchor="center", x=0.5
    )
)
fig.show()

In [11]:
#@title Run this cell to generate the summary table { display-mode: "form" }

df = pd.DataFrame({
    "Model": ["Gemma 270M", "Gemma 1B"],
    "Parameters": [f"{params_270m:,}", f"{params_1b:,}"],
    "Param Memory (float32)": [
        f"{memory_270m:.2f} GB", f"{memory_1b:.2f} GB"
    ],
    "Training Memory (~4x)": [
        f"{memory_270m * 4:.1f} GB", f"{memory_1b * 4:.1f} GB"
    ],
    "Fits 16 GB (inference)?": [
        "✅" if memory_270m < 16 else "❌",
        "✅" if memory_1b < 16 else "❌"
    ],
    "Fits 16 GB (training)?": [
        "✅" if memory_270m * 4 < 16 else "❌",
        "✅" if memory_1b * 4 < 16 else "❌"
    ],
    "Generation Time": [f"{time_270m:.2f}s", f"{time_1b:.2f}s"]
})
df

,Model,Parameters,Param Memory (float32),Training Memory (~4x),Fits 16 GB (inference)?,Fits 16 GB (training)?,Generation Time
0,Gemma 270M,"268,098,176",1.07 GB,4.3 GB,✅,✅,14.42s
1,Gemma 1B,"999,885,952",4.00 GB,16.0 GB,✅,✅,21.16s


### Reflect on the charts and summary table above:

- The 1B model uses roughly 4x more memory than the 270M model,
  but is the output 4x better? What does this tell you
  about the relationship between scale and quality?
- Look at the training memory bars relative to the 16 GB line.
  If neither model fits for training in float32,
  what would you need to change to make training possible?



---

## Section 2: Estimate Training FLOPs

⏱️ **10 minutes** | 🔮 Predict → ▶️ Run → 🔍 Interpret

**The question:** How much compute does it actually cost to train
a model like the ones you just compared?

In the previous section, you saw that bigger models produce better output.
But better output comes at a cost: not just memory, but compute.
In this section, you will implement the standard formula for estimating
training FLOPs for transformer models and use it to compare costs.

### Key concepts

**FLOPs vs FLOPS:**
- **FLOPs** (Floating Point Operations): a count of total arithmetic work.
- **FLOPS** (Floating Point Operations Per Second): a rate measuring hardware speed.

**Training FLOPs for transformers:**
- Forward pass: $2 \times N \times D$ FLOPs
- Backward pass: $4 \times N \times D$ FLOPs (roughly 2x the forward pass)
- **Total: $6 \times N \times D$ FLOPs**
- $N$ = number of parameters, $D$ = number of training tokens

> **📝 Before you write any code**, predict:
>
> - If you double the number of parameters, how does the training cost change?
> - If you double the number of training tokens, how does the training cost change?
> - Which costs more: training a 1B model on 100B tokens,
>   or training a 270M model on 400B tokens?

**🔮 Your prediction:** *(double-click to edit)*

- Doubling parameters:
- Doubling tokens:
- Which costs more:

### TODO: Implement the training FLOPs formula

Complete the function below.
It should return the total estimated training FLOPs for a transformer model.

In [13]:
def estimate_training_flops(num_parameters, num_tokens):
    """
    Estimate total training FLOPs for a transformer model.

    Args:
        num_parameters: Number of model parameters (N)
        num_tokens: Number of training tokens (D)

    Returns:
        Total estimated training FLOPs
    """
    # TODO: Implement the 6ND formula
    # Hint: forward = 2 * N * D, backward = 4 * N * D, total = ?

    total_flops =  # Replace this line

    return total_flops

### Test your implementation

In [26]:
#@title Run the cell below to check your function against known values { display-mode: "form" }

tests = [
    {"N": 1e9, "D": 1e11, "expected": 6e20, "label": "1B params, 100B tokens"},
    {"N": 270e6, "D": 1e11, "expected": 1.62e20, "label": "270M params, 100B tokens"},
    {"N": 4e9, "D": 1e11, "expected": 2.4e21, "label": "4B params, 100B tokens"},
]

all_passed = True
for t in tests:
    result = estimate_training_flops(t["N"], t["D"])
    passed = abs(result - t["expected"]) / t["expected"] < 0.01
    status = "✅" if passed else "❌"
    print(f"{status} {t['label']}: {result:.2e} FLOPs (expected {t['expected']:.2e})")
    if not passed:
        all_passed = False

if all_passed:
    print("\n🎉 All tests passed!")
else:
    print("\n⚠️ Some tests failed. Check your formula.")

✅ 1B params, 100B tokens: 6.00e+20 FLOPs (expected 6.00e+20)
✅ 270M params, 100B tokens: 1.62e+20 FLOPs (expected 1.62e+20)
✅ 4B params, 100B tokens: 2.40e+21 FLOPs (expected 2.40e+21)

🎉 All tests passed!


### Apply your formula to the models from Section 1

Now use your function to estimate the training cost
for the models you compared earlier.
We will assume each model is trained on 100 billion tokens,
which is a realistic scale for modern language models.

In [34]:
# 📊 Compare training FLOPs across model sizes
# Just run this cell. Do not modify it.

training_tokens = 100e9  # 100 billion tokens

model_sizes = {
    "Gemma 270M": 270e6,
    "Gemma 1B": 1e9,
    "Gemma 4B": 4e9,
}

print("=" * 60 + "\nTRAINING FLOPs COMPARISON\n" + "=" * 60)
print(f"Training tokens: {training_tokens:.0e}\n")

flops_results = {}
for name, params in model_sizes.items():
    flops = estimate_training_flops(params, training_tokens)
    flops_results[name] = flops
    print(f"{name:12s}  {flops:.2e} FLOPs")

print(f"\nThe 4B model costs {flops_results['Gemma 4B']/flops_results['Gemma 270M']:.1f}x "
      f"more than the 270M model to train on the same data.")

TRAINING FLOPs COMPARISON
Training tokens: 1e+11

Gemma 270M    1.62e+20 FLOPs
Gemma 1B      6.00e+20 FLOPs
Gemma 4B      2.40e+21 FLOPs

The 4B model costs 14.8x more than the 270M model to train on the same data.


### 📊 What if you change the number of training tokens?

The chart below lets you see how FLOPs change
when you vary both model size and training data.

In [41]:
#@title Run this cell to generate the interactive scaling chart { display-mode: "form" }

sizes = np.array([270e6, 500e6, 1e9, 2e9, 4e9, 7e9, 10e9])

token_counts = [10e9, 50e9, 100e9, 500e9, 1e12]
token_labels = ["10B", "50B", "100B", "500B", "1T"]
colors_line = ["#6B4FCF", "#2D1768", "#F7B733", "#D49A1A", "#C05040"]

fig2 = go.Figure()

for tokens, label, color in zip(token_counts, token_labels, colors_line):
    flops = [estimate_training_flops(n, tokens) for n in sizes]
    fig2.add_trace(go.Scatter(
        x=sizes / 1e9, y=flops,
        mode="lines+markers",
        name=f"{label} tokens",
        line=dict(color=color, width=2),
        marker=dict(size=6)
    ))

fig2.update_layout(
    title=dict(
        text="<b>Training FLOPs: Model Size x Training Tokens</b>",
        y=0.97, x=0.5, xanchor="center"
    ),
    xaxis_title="Model Size (billions of parameters)",
    yaxis_title="Training FLOPs",
    yaxis_type="log",
    height=480, width=750,
    template="plotly_white",
    margin=dict(t=100, b=60, l=80, r=30),
    legend=dict(
        orientation="h", yanchor="bottom", y=1.07,
        xanchor="center", x=0.5
    )
)
fig2.show()

**🔍 Reflect:**

- Training FLOPs scale linearly with both parameters and tokens.
  If you have a fixed compute budget, would you rather train
  a bigger model on less data or a smaller model on more data?
- The 4B model costs roughly 15x more FLOPs than the 270M model
  to train on the same data. Is the quality improvement worth that cost?

---

**💡 Key insight:**

- Compute scales linearly with both model size and training data.
- Doubling either one doubles the cost. But compute is only half the story.

> In the next section, you will discover that even if you have enough compute,
> your model might not fit in memory at all.

In [31]:
# 🧹 Free memory before the next section

del gemma_1b
keras.backend.clear_session()
print("✅ Memory cleared. Ready for Section 3.")

✅ Memory cleared. Ready for Section 3.


---

## Section 3: Hitting a Wall

⏱️ **5 minutes** | 🔮 Predict → ▶️ Run → 🔍 Interpret

**The question:** You saw that bigger models produce better output.
The next size up from Gemma 1B is Gemma 4B.
Can you just load it and use it?

In Section 1, both the 270M and 1B models loaded without any issues.
Now you will try to load the 4B model in float32.

> **📝 Before you run the cell below**, predict:
>
> - Will the 4B model load successfully on your T4 GPU (16 GB)?
> - How much memory would 4 billion parameters need in float32?
> - What error do you expect if it does not fit?

**🔮 Your prediction:** *(double-click to edit)*

- Will it load:
- Memory needed:
- Expected error:

In [32]:
# 🔄 Attempt to load Gemma 3 4B in float32
# Just run this cell. Do not modify it.
# Pay attention to what happens.

print("=" * 60 + "\nATTEMPTING: Gemma 3 4B in float32\n" + "=" * 60)
print(f"4B parameters x 4 bytes (float32) = ~{4e9 * 4 / 1e9:.0f} GB just for weights")
print(f"Your GPU has ~16 GB total\n")
print("Loading model...")

try:
    gemma_4b = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_instruct_4b")
    print("✅ Model loaded successfully!")
    gemma_4b.summary()
except Exception as e:
    print(f"\n❌ FAILED!\n")
    print(f"Error type: {type(e).__name__}")
    print(f"Error message: {e}")

ATTEMPTING: Gemma 3 4B in float32
4B parameters x 4 bytes (float32) = ~16 GB just for weights
Your GPU has ~16 GB total

Loading model...

❌ FAILED!

Error type: ValueError
Error message: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 20971520 bytes.


### What just happened?

The model failed to load because it ran out of memory.
Here is why:

- Gemma 3 4B has approximately 4 billion parameters.
- In float32, each parameter takes 4 bytes.
- That is roughly **16 GB** just for the model weights.
- But your T4 GPU only has about **15 GB of usable memory**
  (some is reserved by the framework and operating system).
- There is no room left. The model cannot fit.

And this is only for **inference** (loading and running the model).
If you wanted to **train** it, you would need roughly 4x that:
parameters + optimizer states + gradients + activations = ~64 GB.

> ⁉️ So the question becomes: **how do you make it fit?**
That is what the rest of this notebook is about.

**🔍 Reflect:**

- Was your prediction correct? If you predicted it would load,
  what assumption were you making about memory?
- The model needs ~16 GB for weights alone, but the GPU has ~15 GB usable.
  Even a small margin matters. What does this tell you about the importance
  of knowing your exact memory budget before choosing a model?

---

**💡 Key insight:**

- A model can fail to load even before you try to train it.
- Memory is a hard constraint, not a soft one. If it does not fit, it does not run.

> In the next section, you will learn to calculate exactly how much memory
> a model needs, so you can predict failures like this before they happen.

---

## Section 4: Estimate GPU Memory

⏱️ **15 minutes** | 🔮 Predict → ▶️ Run → 🔍 Interpret

**The question:** In Section 3, the 4B model crashed because it ran out of memory.
But how much memory does a model actually need?
And where does it all go?

In this section, you will learn to calculate GPU memory usage from first principles.
You will break it down into four components and build a function
that predicts memory usage for any model size and precision format.

### 🧠 Recall: What fills GPU memory during training?

| Component | What it stores | Size (float32) |
|---|---|---|
| **Parameters** | The model weights | $N \times 4$ bytes |
| **Optimizer states** | Momentum + variance (Adam stores 2 copies) | $N \times 8$ bytes |
| **Gradients** | One gradient per parameter | $N \times 4$ bytes |
| **Activations** | Intermediate outputs for backpropagation | Varies (batch, seq length) |

For a rough estimate, **training memory is approximately 4x parameter memory**
(ignoring activations, which depend on batch size and sequence length).

### Precision formats

| Format | Bytes per parameter | Relative to float32 |
|---|---|---|
| float32 | 4 bytes | 1x (baseline) |
| bfloat16 / float16 | 2 bytes | 0.5x |
| int8 | 1 byte | 0.25x |

> **📝 Before you write any code**, predict:
>
> - How much total training memory does the 1B model need in float32?
> - How much would the 4B model need for training in float32?
> - If you switched to bfloat16, would the 4B model fit on a 16 GB GPU for inference?

**🔮 Your prediction:** *(double-click to edit)*

- 1B training memory:
- 4B training memory:
- 4B in bfloat16 for inference:

### TODO: Implement the memory estimation function

Complete the function below. It should return a dictionary
with the memory for each component and the total,
all in gigabytes (GB).

In [ ]:
def estimate_gpu_memory(num_parameters, bytes_per_param=4, training=True):
    """
    Estimate GPU memory usage for a transformer model.

    Args:
        num_parameters: Number of model parameters (N)
        bytes_per_param: Bytes per parameter (4 for float32, 2 for bfloat16, 1 for int8)
        training: If True, include optimizer states and gradients

    Returns:
        Dictionary with memory breakdown in GB
    """
    # Parameter memory
    # TODO: Calculate memory for model weights in GB
    param_memory_gb = ___  # Replace this line

    if training:
        # Optimizer states: Adam stores 2 extra values per parameter (momentum + variance)
        # These are always stored in float32 (4 bytes) regardless of model precision
        # TODO: Calculate optimizer memory in GB
        optimizer_memory_gb = ___  # Replace this line

        # Gradients: one gradient per parameter, same precision as the model
        # TODO: Calculate gradient memory in GB
        gradient_memory_gb = ___  # Replace this line
    else:
        optimizer_memory_gb = 0
        gradient_memory_gb = 0

    total_memory_gb = param_memory_gb + optimizer_memory_gb + gradient_memory_gb

    return {
        "parameters": param_memory_gb,
        "optimizer": optimizer_memory_gb,
        "gradients": gradient_memory_gb,
        "total": total_memory_gb,
    }

### Test your implementation

Run the cell below to check your function against known values.

In [36]:
#@title ✅ Run this cell to test your implementation { display-mode: "form" }

tests = [
    {
        "N": 1e9, "bytes": 4, "training": False,
        "expected_total": 4.0,
        "label": "1B, float32, inference"
    },
    {
        "N": 1e9, "bytes": 4, "training": True,
        "expected_total": 16.0,
        "label": "1B, float32, training"
    },
    {
        "N": 4e9, "bytes": 4, "training": False,
        "expected_total": 16.0,
        "label": "4B, float32, inference"
    },
    {
        "N": 4e9, "bytes": 2, "training": False,
        "expected_total": 8.0,
        "label": "4B, bfloat16, inference"
    },
    {
        "N": 4e9, "bytes": 2, "training": True,
        "expected_total": 48.0,
        "label": "4B, bfloat16, training"
    },
]

all_passed = True
for t in tests:
    result = estimate_gpu_memory(t["N"], t["bytes"], t["training"])
    passed = abs(result["total"] - t["expected_total"]) / t["expected_total"] < 0.01
    status = "✅" if passed else "❌"
    print(f"{status} {t['label']}: {result['total']:.1f} GB (expected {t['expected_total']:.1f} GB)")
    if not passed:
        all_passed = False

if all_passed:
    print("\n🎉 All tests passed!")
else:
    print("\n⚠️ Some tests failed. Check your formula.")
    print("Hint: optimizer states use float32 (4 bytes) regardless of model precision.")

✅ 1B, float32, inference: 4.0 GB (expected 4.0 GB)
✅ 1B, float32, training: 16.0 GB (expected 16.0 GB)
✅ 4B, float32, inference: 16.0 GB (expected 16.0 GB)
✅ 4B, bfloat16, inference: 8.0 GB (expected 8.0 GB)
✅ 4B, bfloat16, training: 48.0 GB (expected 48.0 GB)

🎉 All tests passed!


### Apply your function to the models from this notebook

Now use your function to build a complete memory table
for all three Gemma models across different precision formats.

In [30]:
# 📊 Memory breakdown across all models and precision formats
# Just run this cell. Do not modify it.

configs = [
    ("Gemma 270M", 270e6),
    ("Gemma 1B",   1e9),
    ("Gemma 4B",   4e9),
]

precisions = [
    ("float32", 4),
    ("bfloat16", 2),
    ("int8", 1),
]

rows = []
for model_name, params in configs:
    for prec_name, bpp in precisions:
        inf = estimate_gpu_memory(params, bpp, training=False)
        trn = estimate_gpu_memory(params, bpp, training=True)
        rows.append({
            "Model": model_name,
            "Precision": prec_name,
            "Inference (GB)": f"{inf['total']:.1f}",
            "Training (GB)": f"{trn['total']:.1f}",
            "Fits 16 GB (inference)?": "✅" if inf["total"] < 16 else "❌",
            "Fits 16 GB (training)?": "✅" if trn["total"] < 16 else "❌",
        })

df_memory = pd.DataFrame(rows)
df_memory

,Model,Precision,Inference (GB),Training (GB),Fits 16 GB (inference)?,Fits 16 GB (training)?
0,Gemma 270M,float32,1.1,4.3,✅,✅
1,Gemma 270M,bfloat16,0.5,3.2,✅,✅
2,Gemma 270M,int8,0.3,2.7,✅,✅
3,Gemma 1B,float32,4.0,16.0,✅,❌
4,Gemma 1B,bfloat16,2.0,12.0,✅,✅
5,Gemma 1B,int8,1.0,10.0,✅,✅
6,Gemma 4B,float32,16.0,64.0,❌,❌
7,Gemma 4B,bfloat16,8.0,48.0,✅,❌
8,Gemma 4B,int8,4.0,40.0,✅,❌


**📝 Key takeaways from this table:**
- The 270M model fits for both inference and training in all precision formats. This is the only model you can train on a 16 GB GPU without any optimisation tricks.
- The 1B model fits for inference in all formats, but only fits for training in bfloat16 or int8, not float32.
- Switching from float32 to bfloat16 makes the 4B model fit for inference (8 GB vs 16 GB), but no precision format alone makes it fit for training on a 16 GB GPU.

### 📊 Where does training memory go?

The chart below shows the memory breakdown for the 4B model
across all three precision formats.
This helps you see which component dominates
and why switching precision helps.

In [31]:
#@title Run this cell to generate the memory breakdown chart { display-mode: "form" }

categories = ["float32", "bfloat16", "int8"]
params_mem = []
optim_mem = []
grad_mem = []

for prec_name, bpp in precisions:
    result = estimate_gpu_memory(4e9, bpp, training=True)
    params_mem.append(result["parameters"])
    optim_mem.append(result["optimizer"])
    grad_mem.append(result["gradients"])

fig = go.Figure()

fig.add_trace(go.Bar(
    name="Parameters", x=categories, y=params_mem,
    marker_color="#2D1768", text=[f"{v:.1f}" for v in params_mem],
    textposition="inside"
))
fig.add_trace(go.Bar(
    name="Optimizer states", x=categories, y=optim_mem,
    marker_color="#6B4FCF", text=[f"{v:.1f}" for v in optim_mem],
    textposition="inside"
))
fig.add_trace(go.Bar(
    name="Gradients", x=categories, y=grad_mem,
    marker_color="#F7B733", text=[f"{v:.1f}" for v in grad_mem],
    textposition="inside"
))

fig.add_shape(
    type="line", x0=-0.5, x1=2.5, y0=15, y1=15,
    line=dict(color="red", width=3, dash="dash")
)
fig.add_annotation(
    x=-0.3, y=15, text="<b>T4 GPU limit (~15 GB)</b>",
    showarrow=False, font=dict(color="red", size=12),
    yshift=15, xanchor="left"
)

fig.update_layout(
    title=dict(
        text="<b>Gemma 4B: Training Memory Breakdown by Precision</b>",
        y=0.97, x=0.5, xanchor="center"
    ),
    barmode="stack",
    xaxis_title="Precision Format",
    yaxis_title="Memory (GB)",
    height=500, width=700,
    template="plotly_white",
    margin=dict(t=100, b=60, l=60, r=30),
    legend=dict(
        orientation="h", yanchor="bottom", y=1.02,
        xanchor="center", x=0.5
    )
)
fig.show()

**📝 Key takeaway:** The optimizer states (light purple) dominate training memory at 32 GB regardless of precision, because Adam always stores in float32. Reducing model precision shrinks parameters and gradients but barely dents the total.

### Now you can explain Section 3

Go back to the error you got in Section 3.
Using your function, verify the calculation:

In [41]:
# 📊 Verify the Section 3 failure
# Just run this cell. Do not modify it.

result_4b_f32 = estimate_gpu_memory(4e9, bytes_per_param=4, training=False)

print("=" * 60 + "\nWHY THE 4B MODEL FAILED TO LOAD\n" + "=" * 60)
print(f"Parameter memory (float32):  {result_4b_f32['parameters']:.1f} GB")
print(f"T4 GPU usable memory:        ~15 GB")
print(f"Difference:                  {15 - result_4b_f32['parameters']:.1f} GB")
print(f"\nVerdict: {'❌ Does not fit' if result_4b_f32['parameters'] >= 15 else '✅ Fits'}")
print(f"\nNote: even if it barely fit for inference, training would need")

result_4b_f32_train = estimate_gpu_memory(4e9, bytes_per_param=4, training=True)
print(f"~{result_4b_f32_train['total']:.0f} GB, which is {result_4b_f32_train['total']/15:.1f}x your GPU capacity.")

print(f"\nBut in bfloat16 for inference:")
result_4b_bf16 = estimate_gpu_memory(4e9, bytes_per_param=2, training=False)
print(f"Parameter memory (bfloat16): {result_4b_bf16['parameters']:.1f} GB")
print(f"Verdict: {'✅ Fits!' if result_4b_bf16['parameters'] < 15 else '❌ Does not fit'}")

WHY THE 4B MODEL FAILED TO LOAD
Parameter memory (float32):  16.0 GB
T4 GPU usable memory:        ~15 GB
Difference:                  -1.0 GB

Verdict: ❌ Does not fit

Note: even if it barely fit for inference, training would need
~64 GB, which is 4.3x your GPU capacity.

But in bfloat16 for inference:
Parameter memory (bfloat16): 8.0 GB
Verdict: ✅ Fits!


**🔍 Reflect:**

- Look at the stacked bar chart. The optimizer states take up
  a large share of training memory. Why does Adam need
  2 extra values per parameter, and why are they always in float32?
- The table shows that the 4B model fits on a 16 GB GPU for inference
  in bfloat16 but not in float32. In the next section,
  you will actually load it in bfloat16 and see if it works.

---

**💡 Key insight:**

- You can now predict whether a model will fit on a given GPU before you try to load it.
- The 4B model failed in float32, but the maths shows it should work in bfloat16.

> In the next section, you will test that prediction.

---

## Section 5: Precision Formats and Quantization

⏱️ **10 minutes** | 🔮 Predict → ▶️ Run → 🔍 Interpret

**The question:** In Section 3, the 4B model crashed in float32.
In Section 4, you calculated that bfloat16 cuts parameter memory in half.
But what exactly is bfloat16, why does it work,
and what do you lose by using it?

### 🧠 Quick Recap: How numbers are stored

float32 and bfloat16 are two ways of representing decimal numbers.
Every floating point number has three parts:

| Component | Purpose | float32 | bfloat16 |
|---|---|---|---|
| Sign | Positive or negative | 1 bit | 1 bit |
| Exponent | Range (how big or small the number can be) | 8 bits | 8 bits |
| Mantissa | Precision (how many decimal places) | 23 bits | 7 bits |
| **Total** | | **32 bits (4 bytes)** | **16 bits (2 bytes)** |

bfloat16 keeps the same exponent as float32,
so it can represent the same range of numbers.
It only reduces the mantissa, which means slightly less precision.
For neural network weights, this small loss of precision
rarely affects output quality.

### 🧠 Quick Recap: What is Quantization?

Quantization is the process of reducing the precision of model weights
to save memory. Common formats:

| Format | Bytes per parameter | Use case |
|---|---|---|
| float32 | 4 | Full precision training and inference |
| bfloat16 | 2 | Reduced precision training and inference |
| int8 | 1 | Inference only (too low precision for training) |

Each step down halves the memory.
The trade-off is always precision vs memory.

> **📝 Before you run the next cell**, predict:
>
> - If you switch a 4B model from float32 to bfloat16,
>   how many GB of parameter memory do you save?
> - At what point does reducing precision start to hurt output quality?

**🔮 Your prediction:** *(double-click to edit)*

- Memory saved:
- Quality threshold:

### TODO: Implement a precision comparison function

Complete the function below. Given a model size,
it should return the memory for each precision format
and how much you save by switching.

In [19]:
def compare_precision(num_parameters):
    """
    Compare memory usage across precision formats.

    Args:
        num_parameters: Number of model parameters

    Returns:
        Dictionary with memory for each format and savings
    """
    # TODO: Calculate parameter memory in GB for each format
    float32_gb = ___   # 4 bytes per parameter
    bfloat16_gb = ___  # 2 bytes per parameter
    int8_gb = ___      # 1 byte per parameter

    return {
        "float32": float32_gb,
        "bfloat16": bfloat16_gb,
        "int8": int8_gb,
        "savings_bf16": float32_gb - bfloat16_gb,
        "savings_int8": float32_gb - int8_gb,
    }

In [21]:
#@title ✅ Run this cell to test your implementation { display-mode: "form" }

tests = [
    {"N": 1e9, "expected_f32": 4.0, "expected_bf16": 2.0, "expected_int8": 1.0, "label": "1B"},
    {"N": 4e9, "expected_f32": 16.0, "expected_bf16": 8.0, "expected_int8": 4.0, "label": "4B"},
]

all_passed = True
for t in tests:
    result = compare_precision(t["N"])
    p1 = abs(result["float32"] - t["expected_f32"]) < 0.01
    p2 = abs(result["bfloat16"] - t["expected_bf16"]) < 0.01
    p3 = abs(result["int8"] - t["expected_int8"]) < 0.01
    passed = p1 and p2 and p3
    status = "✅" if passed else "❌"
    print(f"{status} {t['label']}: float32={result['float32']:.1f}, "
          f"bfloat16={result['bfloat16']:.1f}, int8={result['int8']:.1f} GB")
    if not passed:
        all_passed = False

if all_passed:
    print("\n🎉 All tests passed!")
else:
    print("\n⚠️ Some tests failed.")

✅ 1B: float32=4.0, bfloat16=2.0, int8=1.0 GB
✅ 4B: float32=16.0, bfloat16=8.0, int8=4.0 GB

🎉 All tests passed!


### Apply your function

Use your function to see how much memory you save
for each of the models from this notebook.

In [23]:
#@title Run this cell to generate the comparison table { display-mode: "form" }

model_names = ["270M", "1B", "4B"]
model_params = [270e6, 1e9, 4e9]

rows = []
for name, params in zip(model_names, model_params):
    r = compare_precision(params)
    rows.append({
        "Model": name,
        "float32 (GB)": f"{r['float32']:.1f}",
        "bfloat16 (GB)": f"{r['bfloat16']:.1f}",
        "int8 (GB)": f"{r['int8']:.1f}",
        "Saved (bf16)": f"{r['savings_bf16']:.1f} GB",
        "Saved (int8)": f"{r['savings_int8']:.1f} GB",
    })

pd.DataFrame(rows)

,Model,float32 (GB),bfloat16 (GB),int8 (GB),Saved (bf16),Saved (int8)
0,270M,1.1,0.5,0.3,0.5 GB,0.8 GB
1,1B,4.0,2.0,1.0,2.0 GB,3.0 GB
2,4B,16.0,8.0,4.0,8.0 GB,12.0 GB


### 📊 Which models fit on a T4 GPU at each precision?

The chart below shows parameter memory for each model and precision format.
The red line marks the ~15 GB usable memory on a T4.
Any bar below the line fits for inference.

In [47]:
#@title Run this cell to generate the precision chart { display-mode: "form" }

all_names = ["270M", "1B", "4B", "7B", "12B"]
all_params = [270e6, 1e9, 4e9, 7e9, 12e9]

f32_mem = []
bf16_mem = []
int8_mem = []

for p in all_params:
    r = compare_precision(p)
    f32_mem.append(r["float32"])
    bf16_mem.append(r["bfloat16"])
    int8_mem.append(r["int8"])

fig = go.Figure()

fig.add_trace(go.Bar(
    name="float32", x=all_names, y=f32_mem,
    marker_color="#2D1768",
    text=[f"{v:.1f}" for v in f32_mem], textposition="outside"
))
fig.add_trace(go.Bar(
    name="bfloat16", x=all_names, y=bf16_mem,
    marker_color="#6B4FCF",
    text=[f"{v:.1f}" for v in bf16_mem], textposition="outside"
))
fig.add_trace(go.Bar(
    name="int8", x=all_names, y=int8_mem,
    marker_color="#F7B733",
    text=[f"{v:.1f}" for v in int8_mem], textposition="outside"
))

fig.add_shape(
    type="line", x0=-0.5, x1=4.5, y0=15, y1=15,
    line=dict(color="red", width=3, dash="dash")
)
fig.add_annotation(
    x=-0.3, y=15, text="<b>T4 GPU limit (~15 GB)</b>",
    showarrow=False, font=dict(color="red", size=12),
    yshift=15, xanchor="left"
)

fig.update_yaxes(range=[0, max(f32_mem) * 1.15])

fig.update_layout(
    title=dict(
        text="<b>Parameter Memory by Model Size and Precision (Inference Only)</b>",
        y=0.97, x=0.5, xanchor="center"
    ),
    xaxis_title="Model Size",
    yaxis_title="Parameter Memory (GB)",
    barmode="group",
    height=500, width=850,
    template="plotly_white",
    margin=dict(t=100, b=60, l=60, r=30),
    legend=dict(
        orientation="h", yanchor="bottom", y=1.07,
        xanchor="center", x=0.5
    )
)
fig.show()

**📝 Key takeaways:**

- bfloat16 halves parameter memory with minimal quality loss
  because it keeps the same numerical range as float32
  but reduces precision from 23 bits to 7 bits.
- Quantization can move a model from "does not fit" to "fits."
  The 4B model needs 16 GB in float32 but only 8 GB in bfloat16.
- int8 is typically used for inference only.
  Training requires at least bfloat16 to maintain gradient precision.

**🔍 Reflect:**

- Look at the chart. In bfloat16, the 7B model fits for inference (14 GB)
  but the 12B model does not (24 GB).
  What technique from Section 4 would you combine with bfloat16
  to estimate whether training is feasible?
- Quantization reduces memory but not compute.
  A 4B model in int8 still requires the same number of FLOPs to train
  as a 4B model in float32. Why?

---

**💡 Key insight:**

- Quantization is a powerful tool for fitting larger models into limited GPU memory, especially for inference.
- But training memory is dominated by optimizer states (always float32), so precision alone cannot solve every memory problem.

> In the next section, you will learn gradient accumulation:
> a technique that reduces memory by processing smaller batches.

---

## Section 6: Gradient Accumulation

⏱️ **10 minutes** | 🔮 Predict → ▶️ Run → 🔍 Interpret

**The question:** Training works best with large batch sizes
because the gradient estimates are more stable.
But large batches need more memory.
What if you could get the benefits of a large batch
without the memory cost?

### 🧠 Quick Recap: The problem with batch size

During training, your GPU processes a **batch** of examples at once.
Larger batches give more stable gradient estimates,
which means smoother, more reliable training.
But each example in the batch stores its own **activations**
in memory for backpropagation.

Double the batch size = double the activation memory.

On a GPU with limited memory, you quickly hit a wall:
you want batch_size=8 for training quality,
but you can only fit batch_size=1 or 2.

### The solution: gradient accumulation

Gradient accumulation is a simple trick:

1. Process a **small batch** (e.g. batch_size=1)
2. Compute the gradients but **do not update** the weights yet
3. Repeat for N small batches, **accumulating** the gradients
4. After N steps, **average the accumulated gradients** and update the weights

The result: the model sees the same total number of examples
and gets the same gradient update as if you had used a large batch.
But at any given moment, only one small batch is in memory.

$$\text{Effective batch size} = \text{Batch size} \times \text{Accumulation steps}$$

> **📝 Before you run the next cell**, predict:
>
> - If batch_size=2 and accumulation_steps=4, what is the effective batch size?
> - What is the trade-off? What do you gain and what do you lose?

**🔮 Your prediction:** *(double-click to edit)*

- Effective batch size:
- Trade-off:

### TODO: Implement the gradient accumulation calculator

Complete the function below. It should calculate the effective batch size and compare memory usage between a true large batch and gradient accumulation.

In [48]:
def gradient_accumulation_analysis(
    num_parameters,
    bytes_per_param,
    actual_batch_size,
    accumulation_steps,
    activation_memory_per_sample_gb=0.5,
):
    """
    Compare memory: true large batch vs gradient accumulation.

    Args:
        num_parameters: Number of model parameters
        bytes_per_param: Bytes per parameter
        actual_batch_size: Small batch size that fits in memory
        accumulation_steps: Number of steps to accumulate before updating
        activation_memory_per_sample_gb: Estimated activation memory per sample

    Returns:
        Dictionary with effective batch size and memory comparison
    """
    # TODO: Calculate the effective batch size
    effective_batch_size = ___  # Replace this line

    # Base training memory (parameters + optimizer + gradients)
    base = estimate_gpu_memory(num_parameters, bytes_per_param, training=True)
    base_memory = base["total"]

    # TODO: Activation memory for the small batch
    small_batch_activations = ___  # Replace this line

    # TODO: Activation memory for the true large batch
    large_batch_activations = ___  # Replace this line

    memory_with_accumulation = base_memory + small_batch_activations
    memory_without_accumulation = base_memory + large_batch_activations

    return {
        "effective_batch_size": effective_batch_size,
        "memory_with_accumulation": memory_with_accumulation,
        "memory_without_accumulation": memory_without_accumulation,
        "memory_saved": memory_without_accumulation - memory_with_accumulation,
    }

In [54]:
#@title ✅ Run this cell to test your implementation { display-mode: "form" }

tests = [
    {"bs": 1, "steps": 4, "expected_eff": 4, "label": "batch_size=1, steps=4"},
    {"bs": 2, "steps": 8, "expected_eff": 16, "label": "batch_size=2, steps=8"},
    {"bs": 1, "steps": 1, "expected_eff": 1, "label": "batch_size=1, steps=1 (no accumulation)"},
]

all_passed = True
for t in tests:
    result = gradient_accumulation_analysis(1e9, 4, t["bs"], t["steps"])
    passed = result["effective_batch_size"] == t["expected_eff"]
    status = "✅" if passed else "❌"
    print(f"{status} {t['label']}: effective_batch_size={result['effective_batch_size']} "
          f"(expected {t['expected_eff']})")
    if not passed:
        all_passed = False

if all_passed:
    print("\n🎉 All tests passed!")
else:
    print("\n⚠️ Some tests failed.")

✅ batch_size=1, steps=4: effective_batch_size=4 (expected 4)
✅ batch_size=2, steps=8: effective_batch_size=16 (expected 16)
✅ batch_size=1, steps=1 (no accumulation): effective_batch_size=1 (expected 1)

🎉 All tests passed!


### 📊 Memory savings from gradient accumulation

The chart below shows how gradient accumulation
lets you achieve a large effective batch size
while keeping memory usage constant.
The red line grows linearly (true large batch).
The blue line stays flat (gradient accumulation).

In [56]:
#@title Run this cell to generate the gradient accumulation chart { display-mode: "form" }

accum_steps_range = [1, 2, 4, 8, 16]
mem_with = []
mem_without = []
eff_bs = []

for steps in accum_steps_range:
    r = gradient_accumulation_analysis(1e9, 2, 1, steps)
    mem_with.append(r["memory_with_accumulation"])
    mem_without.append(r["memory_without_accumulation"])
    eff_bs.append(r["effective_batch_size"])

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=eff_bs, y=mem_without,
    mode="lines+markers",
    name="True large batch (no accumulation)",
    marker=dict(size=10, color="#C05040"),
    line=dict(color="#C05040", width=2)
))

fig.add_trace(go.Scatter(
    x=eff_bs, y=mem_with,
    mode="lines+markers",
    name="Gradient accumulation (batch_size=1)",
    marker=dict(size=10, color="#2D1768"),
    line=dict(color="#2D1768", width=2)
))

fig.add_shape(
    type="line", x0=0, x1=17, y0=15, y1=15,
    line=dict(color="red", width=3, dash="dash")
)
fig.add_annotation(
    x=0.5, y=15, text="<b>T4 GPU limit (~15 GB)</b>",
    showarrow=False, font=dict(color="red", size=12),
    yshift=15, xanchor="left"
)

fig.update_layout(
    title=dict(
        text="<b>Memory Usage: True Large Batch vs Gradient Accumulation</b>",
        y=0.97, x=0.5, xanchor="center"
    ),
    xaxis_title="Effective Batch Size",
    yaxis_title="Total Training Memory (GB)",
    height=500, width=750,
    template="plotly_white",
    margin=dict(t=100, b=60, l=60, r=30),
    legend=dict(
        orientation="h", yanchor="bottom", y=1.07,
        xanchor="center", x=0.5
    )
)
fig.show()

**📝 Key takeaway:** The red line (true large batch) grows linearly because each sample adds activation memory. The blue line (gradient accumulation) stays flat because only one small batch is in memory at a time. Same effective batch size, fraction of the memory.

### 📊 The trade-off: memory vs time

Gradient accumulation saves memory but costs time.
Instead of processing 8 samples in parallel,
you process them sequentially across 8 steps.

| Term | Meaning |
|---|---|
| `batch_size` | Number of samples processed in a single forward/backward pass |
| `accum_steps` | Number of passes before updating the weights |
| `effective_batch_size` | `batch_size` x `accum_steps` (total samples per weight update) |

All four configurations below have the same effective batch size of 8.
The difference is how much memory they use and how long they take.

In [60]:
#@title Run this cell to generate the trade-off chart { display-mode: "form" }

configs = [
    "batch_size=8, accum_steps=1",
    "batch_size=4, accum_steps=2",
    "batch_size=2, accum_steps=4",
    "batch_size=1, accum_steps=8"
]
memory = []
rel_time = []

for batch_size, accum_steps in [(8, 1), (4, 2), (2, 4), (1, 8)]:
    r = gradient_accumulation_analysis(1e9, 2, batch_size, accum_steps)
    memory.append(r["memory_with_accumulation"])
    rel_time.append(accum_steps)

colors_4 = ["#C05040", "#D49A1A", "#6B4FCF", "#2D1768"]

fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Training Memory (GB)", "Relative Training Time"),
    horizontal_spacing=0.15
)

fig2.add_trace(go.Bar(
    x=configs, y=memory, marker_color=colors_4,
    text=[f"{v:.1f}" for v in memory], textposition="outside",
    showlegend=False
), row=1, col=1)

fig2.add_shape(
    type="line", x0=-0.5, x1=3.5, y0=15, y1=15,
    line=dict(color="red", width=3, dash="dash"),
    row=1, col=1
)
fig2.add_annotation(
    x=-0.3, y=15, text="<b>T4 GPU limit (~15 GB)</b>",
    showarrow=False, font=dict(color="red", size=11),
    yshift=12, xanchor="left", xref="x", yref="y"
)

fig2.add_trace(go.Bar(
    x=configs, y=rel_time, marker_color=colors_4,
    text=[f"{v}x" for v in rel_time], textposition="outside",
    showlegend=False
), row=1, col=2)

fig2.update_yaxes(range=[0, max(memory) * 1.2], row=1, col=1)
fig2.update_yaxes(range=[0, max(rel_time) * 1.3], row=1, col=2)

fig2.update_layout(
    title=dict(
        text="<b>Same Effective Batch Size (8), Different Trade-offs</b>",
        y=0.97, x=0.5, xanchor="center"
    ),
    height=500, width=900,
    template="plotly_white",
    margin=dict(t=100, b=80, l=50, r=30)
)
fig2.show()

**📝 Key takeaway:**

- Moving from left to right (as batch size and accumulation steps increase), memory drops but training time increases.
- All four produce the same gradient update.
- The engineer's job is to pick the largest batch_size that fits in memory and let accum_steps handle the rest.

**🔍 Reflect:**

- Look at the trade-off chart. If you had a fixed deadline for training, how would you choose the right balance between batch size and accumulation steps?
- Gradient accumulation gives you the same gradient update as a true large batch. But is it truly identical? Think about whether batch normalization statistics would behave the same way.

---

## 🎉 Notebook Complete

You made it through the full journey:

1. **Section 1:** You saw that bigger models produce better output.
2. **Section 2:** You calculated that training cost scales linearly
   with model size and data.
3. **Section 3:** You hit a wall when the 4B model
   did not fit on your GPU.
4. **Section 4:** You learned to calculate exactly
   where memory goes and why it crashed.
5. **Section 5:** You learned that quantization (bfloat16, int8)
   halves or quarters memory with minimal quality loss.
6. **Section 6:** You learned that gradient accumulation
   trades time for memory, letting you simulate large batches.

**The core lesson:** Every technical decision about model size,
precision, and batch size involves a trade-off.
The job of an engineer is not to avoid constraints
but to work within them intelligently.

---

**📚 Reference:** [github.com/google-deepmind/ai-foundations/tree/main/course_7](https://github.com/google-deepmind/ai-foundations/tree/main/course_7)

## 💡 Solutions

- **Try first, check second.** Only look at solutions after you have attempted the activity multiple times. The best way to learn is to debug your code piece by piece, not to copy existing solutions.
- **Debug before you peek.** If you are stuck, try adding print statements to see what your code is doing at every step. This builds a much deeper understanding than reading an answer.
- **Type, do not copy.** If you do consult a solution, do not copy and paste it. Read it, close it, and type it yourself. This forces you to understand where you went wrong.

### Section 2 TODO: Implement the training FLOPs formula

In [32]:
def estimate_training_flops(num_parameters, num_tokens):
    """
    Estimate total training FLOPs for a transformer model.

    Args:
        num_parameters: Number of model parameters (N)
        num_tokens: Number of training tokens (D)

    Returns:
        Total estimated training FLOPs
    """
    # Forward pass: 2 * N * D
    # Backward pass: 4 * N * D
    # Total: 6 * N * D
    total_flops = 6 * num_parameters * num_tokens

    return total_flops

### Section 4 TODO: Implement the memory estimation function

In [29]:
def estimate_gpu_memory(num_parameters, bytes_per_param=4, training=True):
    """
    Estimate GPU memory usage for a transformer model.

    Args:
        num_parameters: Number of model parameters (N)
        bytes_per_param: Bytes per parameter (4 for float32, 2 for bfloat16, 1 for int8)
        training: If True, include optimizer states and gradients

    Returns:
        Dictionary with memory breakdown in GB
    """
    param_memory_gb = num_parameters * bytes_per_param / 1e9

    if training:
        # Adam optimizer states: always float32 (4 bytes) regardless of model precision
        optimizer_memory_gb = num_parameters * 8 / 1e9
        # Gradients: same precision as the model
        gradient_memory_gb = num_parameters * bytes_per_param / 1e9
    else:
        optimizer_memory_gb = 0
        gradient_memory_gb = 0

    total_memory_gb = param_memory_gb + optimizer_memory_gb + gradient_memory_gb

    return {
        "parameters": param_memory_gb,
        "optimizer": optimizer_memory_gb,
        "gradients": gradient_memory_gb,
        "total": total_memory_gb,
    }

### Section 5 TODO: Implement a precision comparison function

In [20]:
def compare_precision(num_parameters):
    float32_gb = num_parameters * 4 / 1e9
    bfloat16_gb = num_parameters * 2 / 1e9
    int8_gb = num_parameters * 1 / 1e9

    return {
        "float32": float32_gb,
        "bfloat16": bfloat16_gb,
        "int8": int8_gb,
        "savings_bf16": float32_gb - bfloat16_gb,
        "savings_int8": float32_gb - int8_gb,
    }

### Section 6 TODO: Implement the gradient accumulation calculator

In [49]:
def gradient_accumulation_analysis(
    num_parameters,
    bytes_per_param,
    actual_batch_size,
    accumulation_steps,
    activation_memory_per_sample_gb=0.5,
):
    effective_batch_size = actual_batch_size * accumulation_steps

    base = estimate_gpu_memory(num_parameters, bytes_per_param, training=True)
    base_memory = base["total"]

    small_batch_activations = actual_batch_size * activation_memory_per_sample_gb
    large_batch_activations = effective_batch_size * activation_memory_per_sample_gb

    memory_with_accumulation = base_memory + small_batch_activations
    memory_without_accumulation = base_memory + large_batch_activations

    return {
        "effective_batch_size": effective_batch_size,
        "memory_with_accumulation": memory_with_accumulation,
        "memory_without_accumulation": memory_without_accumulation,
        "memory_saved": memory_without_accumulation - memory_with_accumulation,
    }